In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [2]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

## Handle Missing Values

In [3]:
for name, df in datasets.items():
    print("\n", name)
    print(df.isnull().sum()[df.isnull().sum() > 0])


 customers
Series([], dtype: int64)

 geolocation
Series([], dtype: int64)

 order_items
Series([], dtype: int64)

 order_payments
Series([], dtype: int64)

 order_reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

 orders


order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

 products
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

 sellers
Series([], dtype: int64)

 category_translation
Series([], dtype: int64)


In [4]:

order_reviews["review_comment_title"] = order_reviews["review_comment_title"].fillna("No title")
order_reviews["review_comment_message"] = order_reviews["review_comment_message"].fillna("No comment")

products["product_category_name"] = products["product_category_name"].fillna("Unknown")
products["product_name_lenght"] = products["product_name_lenght"].fillna(0)
products["product_description_lenght"] = products["product_description_lenght"].fillna(0)
products["product_photos_qty"] = products["product_photos_qty"].fillna(0)

products["product_weight_g"] = products["product_weight_g"].fillna(products["product_weight_g"].median())
products["product_length_cm"] = products["product_length_cm"].fillna(products["product_length_cm"].median())
products["product_height_cm"] = products["product_height_cm"].fillna(products["product_height_cm"].median())
products["product_width_cm"] = products["product_width_cm"].fillna(products["product_width_cm"].median())

In [6]:
for name, df in datasets.items():
    print("\n", name)
    print(df.isnull().sum()[df.isnull().sum() > 0])


 customers
Series([], dtype: int64)

 geolocation
Series([], dtype: int64)

 order_items
Series([], dtype: int64)

 order_payments
Series([], dtype: int64)

 order_reviews
Series([], dtype: int64)

 orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

 products
Series([], dtype: int64)

 sellers
Series([], dtype: int64)

 category_translation
Series([], dtype: int64)


## Handle Duplicates

In [7]:
geolocation = geolocation.drop_duplicates()

In [8]:
print("Geolocation duplicates:", geolocation.duplicated().sum())

Geolocation duplicates: 0


## Correct Data Types

we found in step 4 were date and times are stored as object

In [9]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

order_reviews["review_creation_date"] = pd.to_datetime(order_reviews["review_creation_date"])
order_reviews["review_answer_timestamp"] = pd.to_datetime(order_reviews["review_answer_timestamp"])

In [10]:
for name, df in datasets.items():
    print("\n", name)
    print(df.dtypes)


 customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

 geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

 order_items
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

 order_payments
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

 order_reviews
review_id                          object
order_id                         

## Format Date/Time Columns

In [11]:
print(orders["order_purchase_timestamp"].head())
print(order_items["shipping_limit_date"].head())
print(order_reviews["review_creation_date"].head())

0   2017-10-02 10:56:33
1   2018-07-24 20:41:37
2   2018-08-08 08:38:49
3   2017-11-18 19:28:06
4   2018-02-13 21:18:39
Name: order_purchase_timestamp, dtype: datetime64[ns]
0   2017-09-19 09:45:35
1   2017-05-03 11:05:13
2   2018-01-18 14:48:30
3   2018-08-15 10:10:18
4   2017-02-13 13:57:51
Name: shipping_limit_date, dtype: datetime64[ns]
0   2018-01-18
1   2018-03-10
2   2018-02-17
3   2017-04-21
4   2018-03-01
Name: review_creation_date, dtype: datetime64[ns]


In [12]:
orders["order_purchase_date"] = orders["order_purchase_timestamp"].dt.date
orders["order_purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_purchase_month"] = orders["order_purchase_timestamp"].dt.month

## Standardize Categorical Values.

In [14]:
print("Order Status:")
print(orders["order_status"].unique())

print("\nPayment Type:")
print(order_payments["payment_type"].unique())

print("\nCustomer State:")
print(customers["customer_state"].unique())

print("\nSeller State:")
print(sellers["seller_state"].unique())

Order Status:
['delivered' 'invoiced' 'shipped' 'processing' 'unavailable' 'canceled'
 'created' 'approved']

Payment Type:
['credit_card' 'boleto' 'voucher' 'debit_card' 'not_defined']

Customer State:
['SP' 'SC' 'MG' 'PR' 'RJ' 'RS' 'PA' 'GO' 'ES' 'BA' 'MA' 'MS' 'CE' 'DF'
 'RN' 'PE' 'MT' 'AM' 'AP' 'AL' 'RO' 'PB' 'TO' 'PI' 'AC' 'SE' 'RR']

Seller State:
['SP' 'RJ' 'PE' 'PR' 'GO' 'SC' 'BA' 'DF' 'RS' 'MG' 'RN' 'MT' 'CE' 'PB'
 'AC' 'ES' 'RO' 'PI' 'MS' 'SE' 'MA' 'AM' 'PA']


The categorical columns were checked for inconsistent capitalization and category formatting. No obvious inconsistencies were found.

## Handle Invalid Records

In [15]:
print("Review Scores:", order_reviews["review_score"].unique())

Review Scores: [4 5 1 3 2]


In [16]:
print("Invalid Review Scores:",
      order_reviews[~order_reviews["review_score"].between(1, 5)])

Invalid Review Scores: Empty DataFrame
Columns: [review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]
Index: []


there are no invalid review scores. All review scores are between 1 and 5.

In [17]:
print("Invalid Price:", (order_items["price"] < 0).sum())

print("Invalid Freight:", (order_items["freight_value"] < 0).sum())

print("Invalid Payment:", (order_payments["payment_value"] < 0).sum())

print("Invalid Installments:",
      (order_payments["payment_installments"] <= 0).sum())

Invalid Price: 0
Invalid Freight: 0
Invalid Payment: 0
Invalid Installments: 2


In [18]:
print(order_payments[order_payments["payment_installments"] <= 0])

                               order_id  payment_sequential payment_type  \
46982  744bade1fcf9ff3f31d860ace076d422                   2  credit_card   
79014  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card   

       payment_installments  payment_value  
46982                     0          58.69  
79014                     0         129.94  


In [19]:
order_payments = order_payments[
    order_payments["payment_installments"] > 0
]

In [20]:
print((order_payments["payment_installments"] <= 0).sum())

0


product_name_lenght --> product_name_length, product_description_lenght --> product_description_length

In [23]:
products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)

In [24]:
print(products.columns)

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')


In [25]:
datasets["geolocation"] = geolocation
datasets["order_payments"] = order_payments
datasets["products"] = products

sending the cleaned dataset to cleaned folder

In [26]:
import os

os.makedirs("../data/cleaned", exist_ok=True)

for name, df in datasets.items():
    df.to_csv(f"../data/cleaned/{name}.csv", index=False)

print("All cleaned datasets saved successfully.")

All cleaned datasets saved successfully.


In [27]:
import os

print(os.listdir("../data/cleaned"))

['category_translation.csv', 'customers.csv', 'geolocation.csv', 'orders.csv', 'order_items.csv', 'order_payments.csv', 'order_reviews.csv', 'products.csv', 'sellers.csv']
